# 02 - Q-learning training

Train a `QLearningAgent` through self-play against the dealer, watch it learn, and inspect its policy.
Training is seeded, so every number here is reproducible.

In [ ]:
from blackjack.ai.base import Action, GameState
from blackjack.ai.q_learning import QLearningAgent
from blackjack.ai.training import train

## One Q-value update

`Q(s,a) += lr * (reward + discount * max Q(s') - Q(s,a))`. From zero with reward 1 and `lr = 0.1`, the value becomes 0.1.

In [ ]:
demo = QLearningAgent(learning_rate=0.1)
state = GameState(15, 10, False)
print(demo.update(state, Action.HIT, 1))
print(demo.q_table[state])

## Train

Train in chunks and print the win rate of each chunk. It includes exploration moves (epsilon = 0.1), so it understates the greedy policy.

In [ ]:
agent = QLearningAgent()
for chunk in range(5):
    result = train(10_000, seed=chunk, agent=agent)
    print(f"games {chunk * 10_000 + 1:>6}-{(chunk + 1) * 10_000:>6}: "
          f"win rate {result.win_rate:.1%}, states learned {len(agent.q_table)}")

## Inspect what it learned

In [ ]:
for state in [GameState(10, 6, False), GameState(15, 10, False), GameState(17, 6, False), GameState(18, 10, True)]:
    q = agent.q_table[state]
    print(f"{state!s:<56} hit={q[Action.HIT]:+.3f}  stand={q[Action.STAND]:+.3f}  -> {agent.decide(state).value}")

### Policy for hard hands

Rows are the player's score, columns the dealer's up-card (11 = Ace). H = hit, S = stand, `.` = never seen.

In [ ]:
dealer_cards = list(range(2, 12))
print("      " + " ".join(f"{d:>2}" for d in dealer_cards))
for score in range(8, 21):
    row = []
    for d in dealer_cards:
        s = GameState(score, d, False)
        row.append(" ." if s not in agent.q_table else (" H" if agent.decide(s) is Action.HIT else " S"))
    print(f"{score:>4}  " + " ".join(row))

## Save and reload

A saved Q-table reloads into an agent that makes identical decisions. Use `blackjack train` to write `models/q_table.json`.

In [ ]:
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "q_table.json"
agent.save(path)
reloaded = QLearningAgent.load(path)
print("tables equal:", reloaded.q_table == agent.q_table)